In [ ]:
cons = {
    "min_price": 5.00,
    "min_volume": 15
}

In [ ]:
reader_params = ('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench')

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
import statsmodels.api as sm
from scipy.optimize import minimize
import os
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
from typing import List, Dict, Any, Optional

In [ ]:
class Utils:
  @staticmethod
  def read_data(ticker, start, end, constrains=None):
    try:
      t = yf.Ticker(ticker)
      df = t.history(start=start, end=end, interval="1mo")[['Close', 'Volume']]
      df.index = df.index.tz_localize(None).normalize()

      if not Utils.check_constrains(df["Close"].mean(), df["Volume"].mean(), constrains):
        return None, False

      annual_bs = t.balance_sheet

      target_rows = ['Stockholders Equity', 'Share Issued']
      financials = annual_bs.loc[annual_bs.index.intersection(target_rows)].T

      financials.index = pd.to_datetime(financials.index).tz_localize(None).normalize()

      df = pd.merge_asof(
        df.sort_index(),
        financials.sort_index(),
        left_index=True,
        right_index=True,
        direction='backward'
      )

      df["Market_Cap"] = df["Share Issued"] * df["Close"]

      df["BMV"] = df["Stockholders Equity"] / df["Market_Cap"]

      df["Size"] = np.log(df["Market_Cap"])

      df["Momentum"] = df["Close"].pct_change(periods=8)

      df["Returns"] = df["Close"].pct_change()

      df = df[["Size", "BMV", "Momentum", "Returns"]]

      return df, True

    except Exception as e:
      raise Exception(f"Error retrieving data for {ticker}: {e}")


  def check_constrains(close_mean, volume_mean, constrains):
    volume_mean = volume_mean / 1000000

    if constrains is None:
      return True

    if close_mean < constrains["min_price"] or volume_mean < constrains["min_volume"]:
      return False

    return True

  @staticmethod
  def compute_z_scores(x):
    mu = np.mean(x)
    sigma = np.std(x)

    z = (x - mu) / sigma

    return z, sigma

  @staticmethod
  def dot_product(A, B):
    return np.sum(A * B, axis=1).reshape(-1, 1)



In [ ]:
class DataReader:
  def __init__(self, constrains, debug=False):
    self.factors = None
    self.constrains = constrains
    self.market_data = None
    self.debug = debug

  def read_factors(self, reader_params, start, end):
    factors = pdr.DataReader(*reader_params)
    factor_data = factors[0]
    factor_data = factor_data[start:end]
    return factor_data

  def load_marketdata(self, tickers, start, end):
    ticker_data = {}
    for ticker in tickers:
      res, cons_check = Utils.read_data(ticker, start, end, self.constrains)

      if not cons_check:
        print(f"Failed to load: {ticker:^4} | Ticker did not meet the set constrains")
        continue

      ticker_data[ticker] = res

      print(f"---------------- {ticker:^4} ----------------")
      if self.debug:
        print(ticker_data[ticker].head(3))

    full_df = pd.concat(ticker_data, axis=1, names=['Ticker', 'Feature'])
    full_df = full_df.swaplevel(0, 1, axis=1).sort_index(axis=1)

    return full_df.dropna()

  def generate_data(self, tickers, start, end, reader_params):
    factors = self.read_factors(reader_params, start, end)
    market_data = self.load_marketdata(tickers, start, end)

    return market_data, factors


In [ ]:
class FeatureStore(DataReader):
  def __init__(self, constrains, debug:bool=False):
    super().__init__(constrains, debug=False)
    self.debug = debug

  def compute_scores(self, features, returns):
    def get_corr_i(group, target):
        return group.corrwith(target, axis=1)

    scores = features.groupby(level="Feature", axis=1, group_keys=False).apply(
        lambda x: x.sub(x.mean(axis=1), axis=0).div(x.std(axis=1), axis=0)
    )

    forward_returns = returns.shift(1)
    IC = features.groupby(level="Feature", axis=1, group_keys=False).apply(
        get_corr_i, target=forward_returns
    )

    return scores.mul(IC, axis=1, level="Feature")

  def construct_factors(self, returns, factors):
    if isinstance(factors.index, pd.PeriodIndex):
        factors.index = factors.index.to_timestamp()

    rf = factors["RF"]

    factor_returns = factors.drop(columns=["RF"])

    common_index = returns.index.intersection(rf.index).intersection(factor_returns.index)

    excess_returns = returns.loc[common_index].sub(rf.loc[common_index], axis=0).dropna()

    return excess_returns, factor_returns.loc[excess_returns.index]

  def load_or_generate_data(self, tickers, start, end, reader_params):
    returns_path = f"returns_{start}_{end}.parquet"
    factors_path = f"factors_{start}_{end}.parquet"
    scores_path = f"scores{start}_{end}.parquet"

    if os.path.exists(returns_path) and os.path.exists(factors_path) and os.path.exists(scores_path) and not self.debug:
      if self.debug:
        print("loading data")

      scores = pd.read_parquet(scores_path)
      returns = pd.read_parquet(returns_path)
      factors = pd.read_parquet(factors_path)

      if self.debug:
        print(scores.head())
        print(returns.head())
        print(factors.head())

    else:
      if self.debug:
        print("generating data")

      market_data, factors_raw = self.generate_data(tickers, start, end, reader_params)

      features = market_data[["Size", "Momentum", "BMV"]]
      returns_raw = market_data["Returns"].shift()

      scores = self.compute_scores(features=features, returns=returns_raw)

      returns, factors = self.construct_factors(returns=returns_raw, factors=factors_raw)

      scores.to_parquet(scores_path)
      returns.to_parquet(returns_path)
      factors.to_parquet(factors_path)

    return scores, returns, factors


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

class FactorExposureModel:
    def __init__(self, ridge_penalty=10.0, variance_smoothing_factor=0.94,
                 window_size=12, solver="auto", min_variance_floor=None):
      self.ridge_penalty = ridge_penalty
      self.window_size = window_size
      self.solver = solver
      self.variance_smoothing_factor = variance_smoothing_factor
      self.min_variance_floor = min_variance_floor

    def calculate_window_exposure(self, x_window, y_window):
      model = Ridge(alpha=self.ridge_penalty, fit_intercept=True, solver=self.solver)
      model.fit(x_window, y_window)

      beta_exposure = model.coef_
      intercept_alpha = model.intercept_

      squared_residuals = (y_window - model.predict(x_window))**2

      running_variance = squared_residuals.iloc[0]

      for t in range(1, len(squared_residuals)):
        running_variance = (self.variance_smoothing_factor * squared_residuals.iloc[t] +
                            (1 - self.variance_smoothing_factor) * running_variance)

      if self.min_variance_floor:
        running_variance = max(running_variance, self.min_variance_floor)

      return (intercept_alpha, beta_exposure, running_variance)

    def fit(self, factor_data, returns_data):
      factor_names = factor_data.columns
      ticker_names = returns_data.columns
      n_periods = len(returns_data)

      np_betas = np.full((n_periods, len(ticker_names) * len(factor_names)), np.nan)
      np_intercepts = np.full((n_periods, len(ticker_names)), np.nan)
      np_idio_vars = np.full((n_periods, len(ticker_names)), np.nan)

      beta_cols = pd.MultiIndex.from_product([ticker_names, factor_names], names=['Ticker', 'Factor'])

      for i, ticker in enumerate(ticker_names):
        y_series = returns_data[ticker]

        for j in range(self.window_size - 1, n_periods):
          x_window = factor_data.iloc[j - self.window_size + 1 : j + 1]
          y_window = y_series.iloc[j - self.window_size + 1 : j + 1]

          if y_window.isnull().any():
            continue

          alpha_val, beta_val, var_val = self.calculate_window_exposure(x_window, y_window)

          start_col = i * len(factor_names)
          end_col = start_col + len(factor_names)

          np_betas[j, start_col:end_col] = beta_val
          np_intercepts[j, i] = alpha_val
          np_idio_vars[j, i] = var_val

      betas_df = pd.DataFrame(np_betas, columns=beta_cols, index=returns_data.index).dropna()
      valid_index = betas_df.index

      intercepts_df = pd.DataFrame(np_intercepts, columns=ticker_names, index=returns_data.index).loc[valid_index]
      idiosyncratic_vars_df = pd.DataFrame(np_idio_vars, columns=ticker_names, index=returns_data.index).loc[valid_index]

      return intercepts_df, betas_df, idiosyncratic_vars_df

In [ ]:
import pandas as pd

class FactorPremiaModel:
  def __init__(self, score_conf=0.5, rp_shrinkage=0.4, window=2):
    self.factors = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
    self.score_conf = score_conf
    self.rp_shrinkage = rp_shrinkage
    self.window = window

  def cross_sectional_z(self, df):
    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

  def estimate_rolling_risk_premia(self, factor_returns):
    rolling_mean = factor_returns[self.factors].rolling(window=self.window, min_periods=1).mean()
    return rolling_mean.shift(2) * self.rp_shrinkage

  def compute_expected_return(self, betas, scores, factor_returns):
    rolling_risk_premia = self.estimate_rolling_risk_premia(factor_returns)
    systematic_df = pd.DataFrame(0, index=betas.index, columns=betas.columns)

    for factor in self.factors:
      beta_f = betas.xs(factor, axis=1, level=1)
      f_risk_premia = rolling_risk_premia[factor]
      systematic_df += beta_f.mul(f_risk_premia, axis=0)

    common_indices = scores.index.intersection(systematic_df.index).intersection(rolling_risk_premia.dropna().index)

    z_systematic = self.cross_sectional_z(systematic_df.loc[common_indices])
    z_scores = self.cross_sectional_z(scores.loc[common_indices])

    expected_returns = (self.score_conf * z_scores) + ((1 - self.score_conf) * z_systematic)

    return expected_returns

In [ ]:
from dataclasses import dataclass, field

@dataclass
class OptimizationConstrains:
  investment: float = 1
  max_weight: float = 0.05
  turnover_lim: float = 0.2
  min_weight: float = 0
  beta_dev: float = .1

@dataclass
class OptimizationParams:
  obj_fn = None #copy of the class
  n_assets:int = None
  cost_rate: float = 0.002
  w_prew:np.ndarray=None,
  optim_window:int = 2

@dataclass
class GridParams:
  risk_aversion: list = field(default_factory=lambda: [0.1, 0.5, 1, 2, 5, 10])
  factor_target_strength: list = field(default_factory=lambda: [0, 0.05, 0.1, 0.25, 0.5, 1, 2, 5])
  turnover_penalty: list = field(default_factory=lambda: [0, 0.1, 0.5, 1, 2, 5])
  desired_exp: list = field(default_factory=lambda: [0.2, 0.5, 1.0])

  @property
  def grid(self):
    return {
      "risk_aversion": self.risk_aversion,
      "factor_penalty": self.factor_target_strength,
      "turnover_penalty": self.turnover_penalty,
      "desired_exp": self.desired_exp
    }

In [ ]:
class PortfolioObjectiveLoss:
  def __init__(
      self, risk_aversion, factor_penalty,
      turnover_penalty, betas, factor_premia,
      w_prew, cov, desired_exp, use_factor_penalty=True
    ):

    self.risk_aversion = risk_aversion
    self.factor_penalty = factor_penalty
    self.desired_exp = desired_exp
    self.turnover_penalty = turnover_penalty
    self.use_factor_penalty = use_factor_penalty
    self.betas = betas
    self.factor_premia = factor_premia
    self.cov = cov
    self.w_prew = w_prew

  def magnitude(self, x):
    x2_sum = np.sum(x**2)
    return np.sqrt(x2_sum)

  def __call__(self, x):
    factor_target = self.factor_penalty * (self.magnitude(self.betas.T - self.desired_exp))**2 if self.use_factor_penalty else 0
    turnover_penalty = self.turnover_penalty * (self.magnitude(x - self.w_prev))**2

    return - x.T * self.factor_premia + self.risk_aversion * x  * self.cov * x.T + factor_target + turnover_penalty



In [ ]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint, NonlinearConstraint
from sklearn.base import BaseEstimator, RegressorMixin

class SateliteOptimizer(BaseEstimator, RegressorMixin):
    def __init__(self, obj_class, constrains_container, n_assets,
                 risk_aversion=0.1, factor_penalty=0.1,
                 turnover_penalty=0.1, desired_exp=0.2, cost_rate=0.002):
        self.obj_class = obj_class
        self.constrains_container = constrains_container
        self.n_assets = n_assets
        self.risk_aversion = risk_aversion
        self.factor_penalty = factor_penalty
        self.turnover_penalty = turnover_penalty
        self.desired_exp = desired_exp
        self.cost_rate = cost_rate

    def get_constraints(self, betas, w_prev):
        n = self.n_assets
        w_sum_cons = LinearConstraint(np.ones((1, n)), [1], [1])

        lb_size = [self.constrains_container.min_weight or 0] * n
        ub_size = [self.constrains_container.max_weight or 1] * n

        lb_beta = [1 - self.constrains_container.beta_dev]
        ub_beta = [1 + self.constrains_container.beta_dev]
        beta_cons = LinearConstraint(betas.T, lb_beta, ub_beta)

        return [w_sum_cons, beta_cons]

    def fit(self, X, y=None, **fit_params):
        returns = X['returns']
        betas = X['betas']
        factor_premia = X['factor_premia']
        cov = X['cov']
        w_prev = X.get('w_prev', np.full(self.n_assets, 1/self.n_assets))

        obj_fn = self.obj_class(
            risk_aversion=self.risk_aversion,
            factor_penalty=self.factor_penalty,
            turnover_penalty=self.turnover_penalty,
            desired_exp=self.desired_exp,
            betas=betas,
            factor_premia=factor_premia,
            cov=cov,
            w_prev=w_prev
        )

        constraints = self.get_constraints(betas, w_prev)
        bounds = [(self.constrains_container.min_weight or 0,
                   self.constrains_container.max_weight or 1) for _ in range(self.n_assets)]

        res = minimize(
            fun=obj_fn,
            x0=w_prev,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        if not res.success:
            self.weights_ = w_prev
        else:
            self.weights_ = res.x

        self.w_prev_fit_ = w_prev
        return self

    def score(self, X, y=None):
        returns = X['returns']

        p_returns = returns @ self.weights_

        turnover = np.sum(np.abs(self.weights_ - self.w_prev_fit_))
        net_returns = p_returns - (turnover * self.cost_rate)

        if np.std(net_returns) == 0: return -1e6
        return np.mean(net_returns) / np.std(net_returns) * np.sqrt(252)

In [ ]:
@dataclass
class FactorExpParams:
  ridge_penalty: float = 10.0
  variance_smoothing_factor: float = 0.94
  ridge_window_size: int = 12
  solver: str = "auto"

@dataclass
class FactorPremiaParams:
  score_conf: float = 0.5
  rp_shrinkage: float = 0.4
  window: int = 2


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

class SatelitePortfolioModel:
  def __init__(
      self, factor_premia_params: dict,
      factor_exp_params: dict,
      optimization_params: OptimizationParams,
      grid_params: GridParams,
      optimization_constraints: OptimizationConstrains,
      cov_window: int = 6,
      asset_constraints: dict = None,
      n_splits: int = 5,
      debug: bool = False
  ):
    self.debug = debug
    self.cov_window = cov_window
    self.n_splits = n_splits

    self.grid_params = grid_params

    self.feature_store = FeatureStore(constrains=asset_constraints, debug=self.debug)
    self.factor_exp_model = FactorExposureModel(**factor_exp_params)
    self.premia_model = FactorPremiaModel(**factor_premia_params)

    p = optimization_params
    self.base_optimizer = SateliteOptimizer(
      obj_class=p.obj_fn,
      constrains_container=optimization_constraints,
      n_assets=p.n_assets,
      cost_rate=p.cost_rate,
      w_prew=p.w_prew
    )

  def get_features(self, start: str, end: str, tickers: list, reader_params: tuple):
    scores, returns, factors = self.feature_store.load_or_generate_data(
      tickers=tickers,
      start=start,
      end=end,
      reader_params=reader_params
    )
    return scores, returns, factors

  def get_factor_exp(self, factors, returns):
    return self.factor_exp_model.fit(factors, returns)

  def get_factor_premia(self, betas, scores, factors):
    return self.premia_model.compute_expected_return(betas, scores, factors)

  def calculate_covariance(self, betas, factors, ids_risk):
    cov_list = []
    for i in range(len(betas)):
      if i < self.cov_window:
        cov_list.append(np.eye(betas.shape[1]))
        continue

      betas_i = betas.iloc[i].values.reshape(-1, 1)
      factor_cov = factors.iloc[i-self.cov_window:i].cov().values
      D_i = np.diag(ids_risk.iloc[i].values)

      cov_matrix = betas_i @ factor_cov @ betas_i.T + D_i
      cov_list.append(cov_matrix)

    return cov_list

  def optimize_portfolio(self, scores, returns, factors):
      alphas, betas, err_variance = self.get_factor_exp(factors, returns)
      factor_premia = self.get_factor_premia(betas, scores, factors)
      cov_matrices = self.calculate_covariance(betas, factors, err_variance)

      X_data = {
          "returns": returns,
          "betas": betas,
          "factor_premia": factor_premia,
          "cov": cov_matrices,
          "w_prev": self.base_optimizer.w_prev
      }


      tscv = TimeSeriesSplit(n_splits=self.n_splits)

      grid_search = GridSearchCV(
          estimator=self.base_optimizer,
          param_grid=self.grid_params.grid,
          cv=tscv,
          scoring=None,
          n_jobs=-1
      )

      grid_search.fit(X_data)

      print(f"Best Params Found: {grid_search.best_params_}")

      return grid_search.best_estimator_.weights_

In [ ]:
class CoreFeatureStore:
    def __init__(self, start, end, tickers, satelite_portfolio, interval="1d", constrains=None, debug=False):
        self.constrains = constrains
        self.debug = debug
        self.start = start
        self.end = end
        self.tickers = tickers
        self.interval = interval
        self.satelite = satelite_portfolio

    def download_data(self):
        data = {}

        for ticker in self.tickers:
          print(f"---------------- {ticker:^4} ----------------")
          try:
              t = yf.Ticker(ticker)
              df = t.history(start=self.start, end=self.end, interval=self.interval)[["Close"]]
              if df.empty:
                  continue

              df.index = pd.to_datetime(df.index).tz_localize(None).normalize()

              info = t.info
              total_assets = info.get("totalAssets")
              nav_price = info.get("navPrice") or info.get("previousClose") or info.get("regularMarketPreviousClose")

              if total_assets and nav_price:
                  proxy_shares = total_assets / nav_price
                  if self.debug:
                      print(f"{ticker}: Proxy Shares Calculated: {proxy_shares:,.0f}")
              else:
                  proxy_shares = np.nan
                  if self.debug:
                      print(f"Warning: Could not find AUM data for {ticker}. Market Cap will be NaN.")

              df["Market_Cap"] = df["Close"] * proxy_shares

              data[ticker] = df

          except Exception as e:
              print(f"failed to load {ticker}: {e}")

        if not data:
            raise RuntimeError("No data was loaded for any tickers.")

        sat_df = self.satelite.copy()
        sat_df.index = pd.to_datetime(sat_df.index).tz_localize(None).normalize()
        data["satelite"] = sat_df

        full_df = pd.concat(data, axis=1, names=['Ticker', 'Feature'])
        full_df = full_df.swaplevel(0, 1, axis=1).sort_index(axis=1)

        return full_df

In [ ]:
class HMMViewGenerator:
    def __init__(self, n_components, n_iter=400, tol=1e-4, init_params="stmc", cov_type="diag", min_covar=1e-3, verbosity=False):
        self.n_components = n_components
        self.n_iter = n_iter
        self.tol = tol
        self.init_params = init_params
        self.cov_type = cov_type
        self.verbosity = verbosity

    def fit_row(self, X):
      X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
      if np.all(X == X[0, :]):
          X = X + np.random.normal(0, 1e-6, X.shape)

      best_model = None
      best_score = -np.inf
      n_restarts = 5

      for seed in range(n_restarts):
          try:
              model = GaussianHMM(
                  n_components=self.n_components,
                  covariance_type=self.cov_type,
                  n_iter=self.n_iter,
                  tol=self.tol,
                  verbose=self.verbosity,
                  init_params=self.init_params,
                  random_state=seed
              )
              model.fit(X)

              if np.isnan(model.startprob_).any():
                  continue

              score = model.score(X)
              if score > best_score:
                  best_score = score
                  best_model = model

          except Exception:
              continue

      if best_model is None:
          n_features = X.shape[1]
          return (
              0,
              np.full((self.n_components, self.n_components), 1 / self.n_components),
              np.zeros((self.n_components, n_features)),
              np.zeros((self.n_components, n_features)),
              np.full((X.shape[0], self.n_components), 1 / self.n_components),
              np.zeros(X.shape[0])
          )

      transmat = best_model.transmat_.copy()
      for i, row in enumerate(transmat):
          s = row.sum()
          transmat[i] = row / s if (s > 0 and not np.isnan(s)) else np.full(self.n_components, 1.0 / self.n_components)
      best_model.transmat_ = transmat

      step_log_likelihoods = best_model.score_samples(X)

      return (
          best_model.predict(X)[-1],
          best_model.transmat_,
          best_model.means_,
          best_model.covars_,
          best_model.predict_proba(X),
          step_log_likelihoods
      )

In [ ]:
class BLObjective:
  def __init__(self, cov, bl_returns, risk_aversion):
    self.bl_returns = bl_returns
    self.cov = cov
    self.risk_aversion = risk_aversion

  def __call__(self, x):
    return - x.T * self.bl_returns + 0.5 * self.risk_aversion * x  * self.cov * x.T


In [ ]:
class AdaptiveBayesAllocationModel:
  def __init__(self, tau:float, window:int, min_period_window:int|None=None, rebalancing_period:int=14):
    self.window = window
    self.min_period_window = min_period_window or window
    self.rebalancing_period = rebalancing_period
    self.tau = tau

  def rolling_eq_returns(self, w_mkt_window, risk_aversion_val, cov_window):
    return risk_aversion_val * (cov_window @ w_mkt_window)

  def calculate_risk_aversion(self, mkt_window, rf):
    mean_ret = mkt_window.mean() * 252
    var_ret = mkt_window.var() * 252

    return (mean_ret - rf) / var_ret if var_ret != 0 else 2.0

  def analytical_mu_bl(self, cov, eq_returns, hmm_views, uncertanty_i):
    if uncertanty_i.ndim == 1:
        omega = np.diag(uncertanty_i)
    else:
        omega = uncertanty_i

    omega_inv = np.linalg.inv(omega)
    tau_cov_inv = np.linalg.inv(self.tau * cov)

    pick_mtx = np.identity(len(hmm_views))

    precision_mtx = np.linalg.inv(tau_cov_inv + pick_mtx.T @ omega_inv @ pick_mtx)

    weighted_returns = (tau_cov_inv @ eq_returns + pick_mtx.T @ omega_inv @ hmm_views)

    return precision_mtx @ weighted_returns


  def get_bl_features(self, returns_raw, mkt_cap, mkt_returns, views, uncertanty, rf):
    full_index = returns_raw.index
    view_daily = views.reindex(full_index).ffill()
    unc_daily = uncertanty.reindex(full_index).ffill()

    w_mkt_all = mkt_cap.div(mkt_cap.sum(axis=1), axis=0)
    bl_returns_list = []
    ra_list = []
    valid_indices = []

    for i in range(self.window, len(returns_raw)):
        current_date = returns_raw.index[i]
        current_rf = rf.iloc[i]
        sigma_t = returns_raw.iloc[i - self.window : i].cov().values
        lambda_t = self.calculate_risk_aversion(mkt_returns.iloc[i - self.window : i], rf=current_rf)
        pi_t = lambda_t * (sigma_t @ w_mkt_all.iloc[i].values)

        Q_t = view_daily.iloc[i].values
        omega_t = unc_daily.iloc[i].values

        if pd.isna(Q_t).any():
            mu_bl = pi_t
        else:
            mu_bl = self.analytical_mu_bl(sigma_t, pi_t, Q_t, omega_t)

        bl_returns_list.append(mu_bl)
        ra_list.append(lambda_t)
        valid_indices.append(current_date)

    return (
        pd.DataFrame(bl_returns_list, index=valid_indices, columns=returns_raw.columns),
        pd.Series(ra_list, index=valid_indices, name="risk_aversion")
    )

  def get_constraints(self, return_cols):
    n = len(return_cols)
    w_sum_cons = LinearConstraint(np.ones((1, n)), [1], [1])

    col_0_mtx = np.zeros(n)
    idx = list(return_cols).index("satellite")
    col_0_mtx[idx] = 1
    satellite_con = LinearConstraint(col_0_mtx, 0.1, 0.3)

    return [w_sum_cons, satellite_con]


  def fit(self, X, **fit_features):
    views = X["views"]
    returns = X["returns"]
    mkt_cap = X['mkt_cap']
    mkt_returns = X['mkt']
    rf = X['rf']
    uncertanty = X['uncertanty']

    pick_mtx = np.identity(len(returns.columns))

    bl_returns, risk_aversion = self.get_bl_returns(
        returns_raw=returns,
        mkt_cap=mkt_cap,
        mkt_returns=mkt_returns,
        pick_mtx=pick_mtx,
        uncertanty=uncertanty,
        views=views,
        rf=rf
    )

    cons = self.get_constraints(bl_returns.columns)

    final_w = []

    for i in range(self.window, len(bl_returns), self.rebalancing_period):
      window_returns = bl_returns.iloc[i - self.window : i]
      risk_aversion_i = risk_aversion.iloc[i - self.window : i]
      covs = window_returns.cov()

      obj_fn = BLObjective(
          cov=covs,
          bl_returns=window_returns,
          risk_aversion=risk_aversion_i
      )

      res = minimize(
          fun=obj_fn,
          x0=np.ones(len(bl_returns.columns)),
          method='SLSQP',
          constraints=cons
      )

      final_w.append(res.x if res.success else np.ones(len(bl_returns.columns)) / len(bl_returns.columns))
    final_w_df = pd.DataFrame(final_w, index=bl_returns.index, columns=bl_returns.columns)
    return final_w_df


In [ ]:
class CorePortfolio:
  def __init__(self, hmm_params: Dict[str, Any], bl_params: Dict[str, Any], debug=False):
    self.view_generator = HMMViewGenerator(**hmm_params)
    self.adaptive_bayes = AdaptiveBayesAllocationModel(**bl_params)
    self.window = bl_params["window"]
    self.rebalancing_period = bl_params["rebalancing_period"]
    self.debug = debug

  def log_to_simple_params(self, means_log, vars_log):
    MAX_EXP = 700.0

    exponent_mean = np.clip(means_log + 0.5 * vars_log, -MAX_EXP, MAX_EXP)
    means_simple = np.exp(exponent_mean) - 1

    exponent_var = np.clip(vars_log, -MAX_EXP, MAX_EXP)
    exponent_2m_v = np.clip(2 * means_log + vars_log, -MAX_EXP, MAX_EXP)
    vars_simple = (np.exp(exponent_var) - 1) * np.exp(exponent_2m_v)

    return means_simple, vars_simple

  def generate_view(self, returns_i):
    log_returns = np.log(returns_i + 1)

    state_i, transmat, means, covars, posterior_probs, loglik = self.view_generator.fit_row(log_returns)

    transmat_safe = np.copy(transmat)
    for idx, row in enumerate(transmat_safe):
        row_sum = np.sum(row)
        if row_sum == 0 or np.isnan(row_sum):
            transmat_safe[idx] = np.full(len(row), 1.0 / len(row))
        else:
            transmat_safe[idx] = row / row_sum

    prob_next_states = transmat_safe[state_i]

    covars_arr = np.array(covars)
    if covars_arr.ndim == 3:
      vars_log = np.array([np.diagonal(c) for c in covars_arr])
    else:
      vars_log = covars_arr

    means_log = np.array(means)

    means_simple, vars_simple = self.log_to_simple_params(means_log, vars_log)

    pred_views_simple = prob_next_states @ means_simple

    diff = means_simple - pred_views_simple[np.newaxis, :]
    diff = np.clip(diff, -1e154, 1e154)
    mean_discrepancy = diff ** 2


    var_posterior = np.sum(
      prob_next_states[:, np.newaxis] * (vars_simple + mean_discrepancy),
      axis=0
    )

    try:
        if isinstance(loglik, tuple):
            loglik_iterable = loglik[0]
        else:
            loglik_iterable = loglik

        loglik_flat = np.array([float(np.ravel(item)[0]) if np.size(item) > 0 else 0.0 for item in loglik_iterable])
    except Exception:
        loglik_flat = np.zeros(len(returns_i))

    if len(loglik_flat) == 0:
        loglik_flat = np.zeros(1)

    last_ll = loglik_flat[-1]
    ll_means = np.mean(loglik_flat[:-5]) if len(loglik_flat) > 5 else np.mean(loglik_flat)

    loglik_penalty = np.exp(np.maximum(0.0, ll_means - last_ll))
    uncertainty = var_posterior * loglik_penalty

    return pred_views_simple, uncertainty

  def generate_hmm_features(self, returns):
    views_list = []
    uncertainty_list = []
    indices = []

    for i in range(self.window, len(returns), self.rebalancing_period):
      window_returns = returns.iloc[i - self.window : i]

      view, omega = self.generate_view(window_returns)

      views_list.append(view)
      uncertainty_list.append(omega)
      indices.append(returns.index[i - 1])

    view_cols = [f"view_{c}" for c in returns.columns]
    unc_cols = [f"unc_{c}" for c in returns.columns]

    df_views = pd.DataFrame(views_list, index=indices, columns=view_cols)
    df_unc = pd.DataFrame(uncertainty_list, index=indices, columns=unc_cols)

    return pd.concat([df_views, df_unc], axis=1)

  def _get_demo_rf(self, start, end, reader_params):
    factors = pdr.DataReader(*reader_params)
    factor_data = factors[0]
    factor_data = factor_data[start:end]["RF"]
    return factor_data

  def _generate_satelite(self, start, end, reader_params, satelite, use_demo_sat, interval):
    if use_demo_sat:
      rf = self._get_demo_rf(reader_params=reader_params, start=start, end=end)
      satelite = self._get_demo_satelite(start=start, end=end, satelite=satelite, interval=interval)
    else:
      pass

    return satelite, rf

  def _get_demo_satelite(self, start, end, satelite, interval):
    ticker_obj = yf.Ticker(satelite)
    satelite_portfolio = ticker_obj.history(start=start, end=end, interval=interval)[["Close"]]

    if satelite_portfolio.empty:
        return satelite_portfolio

    satelite_portfolio.index = pd.to_datetime(satelite_portfolio.index).tz_localize(None).normalize()

    info = ticker_obj.info
    quote_type = info.get("quoteType")

    market_cap_proxy = None

    if quote_type == "EQUITY":
        shares = info.get("sharesOutstanding")
        if shares:
            market_cap_proxy = shares
            if self.debug: print(f"Satellite {satelite}: Using Shares Outstanding.")

    elif quote_type in ["ETF", "INDEX"]:
        total_assets = info.get("totalAssets")
        nav_price = info.get("navPrice") or info.get("previousClose")
        if total_assets and nav_price:
            market_cap_proxy = total_assets / nav_price
            if self.debug: print(f"Satellite {satelite}: Using AUM/NAV Proxy Shares.")

    if market_cap_proxy is None:
        market_cap_proxy = info.get("sharesOutstanding") or np.nan

    satelite_portfolio["Market_Cap"] = satelite_portfolio["Close"] * market_cap_proxy

    return satelite_portfolio

  def download_data(
      self, core_tickers:List[str], start:str, end:str, reader_params:tuple[str],
      satelite_tickers:List[str]=None, interval:str="1d", demo_satelite_ticker:Optional[str]=None
    ):
    returns_path = f"core_data_{start}_{end}.parquet"
    satelite = satelite_tickers if satelite_tickers is not None else demo_satelite_ticker
    use_demo_sat = demo_satelite_ticker is not None

    satelite_portfolio, rf = self._generate_satelite(start=start, end=end, reader_params=reader_params, satelite=satelite, use_demo_sat=use_demo_sat, interval=interval)

    if os.path.exists(returns_path):
      core_data = pd.read_parquet(returns_path)

    else:
      core_data = CoreFeatureStore(
        tickers = core_tickers,
        start=start,
        end=end,
        interval=interval,
        satelite_portfolio=satelite_portfolio
      ).download_data()

      core_data.to_parquet(returns_path)

    return core_data, rf

  def fit(self, data, rf):
    returns = data["Close"].pct_change().dropna()
    views = self.generate_hmm_features(returns)
    return views

In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass